In [1]:
# Importamos Path para construir rutas de archivos de forma portable.
from pathlib import Path

# Importamos OpenCV para leer y procesar imágenes.
import cv2

# Importamos NumPy para trabajar con arreglos numéricos.
import numpy as np

# Importamos Matplotlib para mostrar imágenes y gráficos.
import matplotlib.pyplot as plt

# Definimos la ruta de la imagen principal que usaremos en los ejemplos.
ruta_imagen_principal = Path('Imagenes') / 'messi.jpg'

# Definimos la ruta de la imagen que usaremos para segmentación por color.
ruta_imagen_segmentacion = Path('Imagenes') / 'globos.jpg'

# Mostramos la carpeta actual desde la que se ejecuta el cuaderno.
print(f'Carpeta actual: {Path.cwd()}')

# Mostramos la ruta relativa de la imagen principal.
print(f'Imagen principal: {ruta_imagen_principal}')

# Mostramos la ruta completa de la imagen principal.
print(f'Ruta completa imagen principal: {ruta_imagen_principal.resolve()}')


# Mostramos la ruta relativa de la imagen de segmentación.
print(f'Imagen para segmentación: {ruta_imagen_segmentacion}')

# Mostramos la ruta completa de la imagen de segmentación.
print(f'Ruta completa imagen segmentación: {ruta_imagen_segmentacion.resolve()}')

# Verificamos si el archivo de la imagen principal existe.
print(f'Existe imagen principal: {ruta_imagen_principal.exists()}')

# Verificamos si el archivo de la imagen de segmentación existe.
print(f'Existe imagen segmentación: {ruta_imagen_segmentacion.exists()}')

Carpeta actual: c:\Users\carme\OneDrive\Documents\Procesamiento de imágenes\rodriguez-carmen-pdi-1c-2026\005  - computer_vision_parte_1\002 - PRA
Imagen principal: Imagenes\messi.jpg
Ruta completa imagen principal: C:\Users\carme\OneDrive\Documents\Procesamiento de imágenes\rodriguez-carmen-pdi-1c-2026\005  - computer_vision_parte_1\002 - PRA\Imagenes\messi.jpg
Imagen para segmentación: Imagenes\globos.jpg
Ruta completa imagen segmentación: C:\Users\carme\OneDrive\Documents\Procesamiento de imágenes\rodriguez-carmen-pdi-1c-2026\005  - computer_vision_parte_1\002 - PRA\Imagenes\globos.jpg
Existe imagen principal: True
Existe imagen segmentación: True


In [2]:

from ipywidgets import interact, IntSlider

# Definimos la ruta de la imagen principal que usaremos en los ejemplos.
ruta_imagen_principal = Path('Imagenes') / 'messi.jpg'

# Definimos la ruta de la imagen que usaremos para segmentación por color.
ruta_imagen_segmentacion = Path('Imagenes') / 'globos.jpg'

# Leemos la imagen y la convertimos a RGB para mostrarla con Matplotlib.
imagen_bgr = cv2.imread(str(ruta_imagen_principal))
imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

# Convertimos la imagen original a HSV.
imagen_hsv = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2HSV)

def explorar_hsv(delta_h=0, escala_s=100, escala_v=100):
    hsv_mod = imagen_hsv.copy().astype(np.float32)

    # Modificamos el tono.
    hsv_mod[:, :, 0] = (hsv_mod[:, :, 0] + delta_h) % 180

    # Modificamos saturación y brillo.
    hsv_mod[:, :, 1] = np.clip(hsv_mod[:, :, 1] * (escala_s / 100), 0, 255)
    hsv_mod[:, :, 2] = np.clip(hsv_mod[:, :, 2] * (escala_v / 100), 0, 255)

    hsv_mod = hsv_mod.astype(np.uint8)

    # Volvemos a RGB para visualizar el resultado.
    rgb_mod = cv2.cvtColor(hsv_mod, cv2.COLOR_HSV2RGB)

    # Extraemos canales para mostrarlos por separado.
    h, s, v = cv2.split(hsv_mod)

    plt.figure(figsize=(16, 8), constrained_layout=True)

    plt.subplot(2, 3, 1)
    plt.imshow(imagen_rgb)
    plt.title('Imagen original', fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 3, 2)
    plt.imshow(rgb_mod)
    plt.title('Imagen modificada en HSV', fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 3, 3)
    plt.imshow(cv2.cvtColor(hsv_mod, cv2.COLOR_HSV2RGB))
    plt.title('Vista resultante', fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 3, 4)
    plt.imshow(h, cmap='hsv')
    plt.title('Canal H - Tono', fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 3, 5)
    plt.imshow(s, cmap='gray')
    plt.title('Canal S - Saturación', fontweight='bold')
    plt.axis('off')

    plt.subplot(2, 3, 6)
    plt.imshow(v, cmap='gray')
    plt.title('Canal V - Brillo', fontweight='bold')
    plt.axis('off')

    plt.show()

interact(
    explorar_hsv,
    delta_h=IntSlider(min=-90, max=90, step=1, value=0, description='Hue'),
    escala_s=IntSlider(min=0, max=200, step=5, value=100, description='Sat %'),
    escala_v=IntSlider(min=0, max=200, step=5, value=100, description='Val %')
);


interactive(children=(IntSlider(value=0, description='Hue', max=90, min=-90), IntSlider(value=100, description…

In [3]:


# Definimos la ruta de la imagen.
ruta_imagen_principal = Path('Imagenes') / 'globos.jpg'

# Leemos la imagen y la convertimos a RGB para visualizarla correctamente.
imagen_bgr = cv2.imread(str(ruta_imagen_principal))
if imagen_bgr is None:
    raise FileNotFoundError(f'No se pudo leer la imagen: {ruta_imagen_principal}')

imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

# Convertimos la imagen a HSV para trabajar con segmentación por color.
imagen_hsv = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2HSV)

def segmentar_hsv(h_min=0, h_max=179, s_min=0, s_max=255, v_min=0, v_max=255):
    # Definimos los límites inferior y superior del color a detectar.
    limite_inferior = np.array([h_min, s_min, v_min], dtype=np.uint8)
    limite_superior = np.array([h_max, s_max, v_max], dtype=np.uint8)

    # Generamos una máscara binaria con los píxeles dentro del rango HSV.
    mascara = cv2.inRange(imagen_hsv, limite_inferior, limite_superior)

    # Aplicamos la máscara sobre la imagen original.
    resultado = cv2.bitwise_and(imagen_rgb, imagen_rgb, mask=mascara)

    plt.figure(figsize=(15, 5), constrained_layout=True)

    plt.subplot(1, 3, 1)
    plt.imshow(imagen_rgb)
    plt.title('Imagen original', fontweight='bold')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(mascara, cmap='gray')
    plt.title('Máscara HSV', fontweight='bold')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(resultado)
    plt.title('Segmentación por color', fontweight='bold')
    plt.axis('off')

    plt.show()

interact(
    segmentar_hsv,
    h_min=IntSlider(min=0, max=179, step=1, value=0, description='H min'),
    h_max=IntSlider(min=0, max=179, step=1, value=179, description='H max'),
    s_min=IntSlider(min=0, max=255, step=1, value=0, description='S min'),
    s_max=IntSlider(min=0, max=255, step=1, value=255, description='S max'),
    v_min=IntSlider(min=0, max=255, step=1, value=0, description='V min'),
    v_max=IntSlider(min=0, max=255, step=1, value=255, description='V max')
);


interactive(children=(IntSlider(value=0, description='H min', max=179), IntSlider(value=179, description='H ma…